# Generative Modeling with Autoencoders

So far, we have focused on **classification tasks**, where models predict labels.
In this notebook, we move to a different paradigm:
> **Generative models**

Instead of predicting labels, generative models learn how data is distributed,
and can **generate new samples**.

We will explore:
 - Autoencoders (AE)
 - Variational Autoencoders (VAE)
 - Conditional Variational Autoencoders (CVAE)

 The key idea: learn a **latent representation** of data and generate from it

## Section 1: Data Preprocessing

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

transform = transforms.ToTensor()

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transform)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

## Section 2: Autoencoder (AE)

We start with a simple model:

 - Encoder → compresses image into latent vector
 - Decoder → reconstructs image from latent vector

Goal: learn a compact representation of the input

In [ ]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1,28,28))
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

### Training AE

In [ ]:
import torch.optim as optim

model = Autoencoder(latent_dim=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for x, _ in train_loader:
        x = x.to(device)
        
        optimizer.zero_grad()
        x_hat, _ = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}: {total_loss / len(train_loader):.4f}")

### AE Inference (Generation)

We sample from latent space and decode.

In [ ]:
import matplotlib.pyplot as plt

model.eval()

with torch.no_grad():
    z = torch.randn(16, 32).to(device)
    samples = model.decoder(z).cpu()

plt.figure(figsize=(5,5))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(samples[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.show()

**Observation:**

Generated samples are blurry, noisy and meaningless.

Why?
> Latent space is not structured.

Let's fix this.

## Section 3: Variational Autoencoder (VAE)

Idea:
Instead of mapping input to a single point,
we map it to a **distribution (mean + variance)**.

This forces latent space to be continuous and structured, making generation more meaningful.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.fc2 = nn.Linear(latent_dim, 128)
        self.fc3 = nn.Linear(128, 28*28)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

    def forward(self, x):
        x = x.view(x.size(0), -1)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat.view(-1,1,28,28), mu, logvar

### VAE Training
We extend the AE training loop by adding KL (Kullback-Leibler) divergence.

Total loss:
> reconstruction loss + KL divergence

In [ ]:
model = VAE(latent_dim=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def vae_loss(x_hat, x, mu, logvar):
    recon = nn.functional.binary_cross_entropy(
        x_hat.view(x.size(0), -1),
        x.view(x.size(0), -1),
        reduction='sum'
    )
    
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon + kl

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for x, _ in train_loader:
        x = x.to(device)
        
        optimizer.zero_grad()
        
        x_hat, mu, logvar = model(x)
        loss = vae_loss(x_hat, x, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}: {total_loss / len(train_loader):.4f}")

### VAE Inference (Generation)

Now we sample from a structured latent space.

In [ ]:
model.eval()

with torch.no_grad():
    z = torch.randn(16, 32).to(device)
    samples = model.decode(z).cpu()

import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(samples[i].view(28,28), cmap='gray')
    plt.axis('off')

plt.show()

## Section 4: Conditional VAE (CVAE)

Problem:
VAE generates random digits — we have no control.

Solution: condition the model on labels

Now we can generate a **specific digit**.

In [ ]:
class CVAE(nn.Module):
    def __init__(self, latent_dim=32, n_classes=10):
        super().__init__()
        self.n_classes = n_classes
        
        self.fc1 = nn.Linear(28*28 + n_classes, 128)
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        self.fc2 = nn.Linear(latent_dim + n_classes, 128)
        self.fc3 = nn.Linear(128, 28*28)

    def encode(self, x, y):
        y_onehot = torch.nn.functional.one_hot(y, self.n_classes).float()
        x = x.view(x.size(0), -1)
        h = torch.relu(self.fc1(torch.cat([x, y_onehot], dim=1)))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z, y):
        y_onehot = torch.nn.functional.one_hot(y, self.n_classes).float()
        h = torch.relu(self.fc2(torch.cat([z, y_onehot], dim=1)))
        return torch.sigmoid(self.fc3(h)).view(-1,1,28,28)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, y)
        return x_hat, mu, logvar

### Training CVAE

Training is similar to VAE, but now:

- we pass labels to the model
- we condition both encoder and decoder

In [ ]:
model = CVAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def loss_fn(x_hat, x, mu, logvar):
    recon = nn.functional.binary_cross_entropy(
        x_hat.view(x.size(0), -1),
        x.view(x.size(0), -1),
        reduction='sum'
    )
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        x_hat, mu, logvar = model(x, y)
        loss = loss_fn(x_hat, x, mu, logvar)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}: {total_loss / len(train_loader):.4f}")

### CVAE Inference (Generation)

In [ ]:
import torch

def generate(model, y, n_samples=10, latent_dim=32, device="cuda"):
    model.eval()

    with torch.no_grad():
        # 1. sample
        z = torch.randn(n_samples, latent_dim).to(device)

        # 2. repeat label
        y = torch.tensor([y] * n_samples).to(device)

        # 3. decode
        samples = model.decode(z, y)

    return samples


digits = generate(model, y=7, n_samples=16)

def show_grid(images):
    images = images.cpu()
    
    fig, axes = plt.subplots(4, 4, figsize=(5,5))
    
    idx = 0
    for i in range(4):
        for j in range(4):
            axes[i, j].imshow(images[idx].squeeze(), cmap='gray')
            axes[i, j].axis('off')
            idx += 1
    
    plt.show()

show_grid(digits)

## Section 5: Tasks

### Basic Level 

Explore and understand the behavior of AE, VAE, and CVAE:

1. **Sampling Comparison (AE vs VAE)**

      AE:
      - Reconstruction (data → AE → data): `x → encoder → decoder(x)`; compare `x` and `decoder(x)`
      - Random sampling (generation): `z ~ N(0,1) → decoder(z)`
      
      VAE:
      - Reconstruction (data → latent posterior → data): `x → encoder → z ~ q(z|x) → decoder(z)`
      - Random sampling (generative use): `z ~ N(0,1) → decoder(z)`

2. **Latent Dimension**
   - Try different latent_dim values (e.g., 2, 32, 128) and compare results quality

3. **KL Divergence (VAE)**
   - Modify loss: `loss = reconstruction + β * KL`
   - Try β = 0, 0.1, 1

4. **Latent Space Visualization**
   - Use latent_dim = 2
   - Encode dataset into latent space
   - Plot latent vectors colored by class label

---

### Deep Level 

Choose ONE:

**A. New Dataset**
- Apply VAE or CVAE to a different dataset
- Suggested (TorchVision):
  - FashionMNIST
  - KMNIST
  - CIFAR-10

> See available datasets:
https://pytorch.org/vision/stable/datasets.html


**B. Beyond Generation**
- Use autoencoders for a different purpose than generation

Possible directions:
- **anomaly detection** (e.g., train on digits 0–8, test on 9, compare reconstruction error)
- **denoising** (e.g., add random noise to images and train the model to reconstruct clean inputs)
- **representation learning** (e.g., use encoder outputs as features and visualize with PCA or t-SNE)

> Define your own experiment:
- what is the goal?
- how do you evaluate it?
- what do you observe?

